<a href="https://colab.research.google.com/github/DunyaVatandasi/abd-borsa-calismalari/blob/main/02_haber_duyarlilik.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q yfinance transformers

import yfinance as yf
import pandas as pd
from transformers import pipeline

# FinBERT modelini yükle (ilk seferde 1-2 dakika sürebilir)
finbert = pipeline("text-classification", model="ProsusAI/finbert")

def haber_duyarlilik(sembol, adet=10):
    haberler = yf.Ticker(sembol).news[:adet]
    satirlar = []
    for h in haberler:
        icerik = h.get("content", h)
        baslik = icerik.get("title", "")
        if not baslik:
            continue
        sonuc = finbert(baslik)[0]
        satirlar.append({
            "Tarih": icerik.get("pubDate", "")[:10],
            "Kaynak": icerik.get("provider", {}).get("displayName", ""),
            "Başlık": baslik,
            "Duyarlılık": sonuc["label"],
            "Güven": round(sonuc["score"], 2),
        })
    tablo = pd.DataFrame(satirlar)
    print(f"===== {sembol} haber duyarlılığı =====")
    print(tablo["Duyarlılık"].value_counts(), "\n")
    return tablo

pd.set_option("display.max_colwidth", 100)
haber_duyarlilik("AAPL")

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

RuntimeError: *** YAHOO! FINANCE IS CURRENTLY DOWN! ***
Our engineers are working quickly to resolve the issue. Thank you for your patience.

In [2]:
!pip install -q feedparser
import feedparser, urllib.parse

def haberleri_getir(sembol, adet=10):
    # 1) Önce Yahoo Finance'i dene
    try:
        sonuc = []
        for h in yf.Ticker(sembol).news[:adet]:
            icerik = h.get("content", h)
            if icerik.get("title"):
                sonuc.append({
                    "Tarih": icerik.get("pubDate", "")[:10],
                    "Kaynak": icerik.get("provider", {}).get("displayName", ""),
                    "Başlık": icerik["title"],
                })
        if sonuc:
            return sonuc, "Yahoo Finance"
    except Exception:
        pass

    # 2) Yahoo engellerse yedek: Google News
    sorgu = urllib.parse.quote(f"{sembol} stock")
    url = f"https://news.google.com/rss/search?q={sorgu}&hl=en-US&gl=US&ceid=US:en"
    sonuc = []
    for e in feedparser.parse(url).entries[:adet]:
        sonuc.append({
            "Tarih": pd.to_datetime(e.published).strftime("%Y-%m-%d"),
            "Kaynak": e.get("source", {}).get("title", ""),
            "Başlık": e.title.rsplit(" - ", 1)[0],  # sondaki yayıncı adını at
        })
    return sonuc, "Google News"

def haber_duyarlilik(sembol, adet=10):
    haberler, kaynak = haberleri_getir(sembol, adet)
    for h in haberler:
        s = finbert(h["Başlık"])[0]
        h["Duyarlılık"] = s["label"]
        h["Güven"] = round(s["score"], 2)
    tablo = pd.DataFrame(haberler)
    print(f"===== {sembol} haber duyarlılığı (kaynak: {kaynak}) =====")
    print(tablo["Duyarlılık"].value_counts(), "\n")
    return tablo

haber_duyarlilik("AAPL")


===== AAPL haber duyarlılığı (kaynak: Google News) =====
Duyarlılık
neutral     8
negative    2
Name: count, dtype: int64 



,Tarih,Kaynak,Başlık,Duyarlılık,Güven
0,2026-11-14,Yahoo Finance,Is Apple Stock a Buy Right Now?,neutral,0.91
1,2026-11-11,CNN,AAPL Stock Quote Price and Forecast,neutral,0.94
2,2026-11-14,The Motley Fool,Is Apple Stock a Buy Right Now?,neutral,0.91
3,2026-11-11,Barron's,Why the iPhone Duo Could Kick Off a ‘New Decade’ for Apple,neutral,0.66
4,2026-11-13,Yahoo Finance,What Could Still Send Apple Stock Higher?,negative,0.70
5,2026-11-11,Benzinga,What's Going On With Apple Stock Wednesday?,negative,0.55
6,2026-11-13,The Motley Fool,"Apple Is the World’s Largest Company, and Nvidia Is Second. Here’s What $10,000 Invested in Each...",neutral,0.94
7,2026-11-11,Yahoo Finance,Apple Shares (AAPL) Look Expensive Heading into 2027,neutral,0.75
8,2026-11-09,The Motley Fool,Where Will Apple Stock Be in 5 Years?,neutral,0.93
9,2026-11-09,Yahoo Finance,Where Will Apple Stock Be in 5 Years?,neutral,0.93


In [3]:
import requests

def stocktwits_duyarlilik(sembol):
    url = f"https://api.stocktwits.com/api/2/streams/symbol/{sembol}.json"
    basliklar = {"User-Agent": "Mozilla/5.0"}  # kendimizi tarayıcı gibi tanıtıyoruz

    try:
        yanit = requests.get(url, headers=basliklar, timeout=15)
    except Exception as e:
        print(f"Bağlantı kurulamadı: {e}")
        return None

    if yanit.status_code != 200:
        print(f"StockTwits erişimi reddetti (kod: {yanit.status_code}).")
        return None

    satirlar = []
    for m in yanit.json().get("messages", []):
        etiket = (m.get("entities", {}).get("sentiment") or {}).get("basic")
        satirlar.append({
            "Tarih": m.get("created_at", "")[:16].replace("T", " "),
            "Kullanıcı": m.get("user", {}).get("username", ""),
            "Takipçi": m.get("user", {}).get("followers", 0),
            "Etiket": etiket or "Etiketsiz",
            "Mesaj": m.get("body", ""),
        })

    tablo = pd.DataFrame(satirlar)
    sayim = tablo["Etiket"].value_counts()
    boga = sayim.get("Bullish", 0)
    ayi = sayim.get("Bearish", 0)

    print(f"===== {sembol} StockTwits duyarlılığı (son {len(tablo)} mesaj) =====")
    print(sayim, "\n")
    if boga + ayi > 0:
        print(f"Boğa oranı (etiketli mesajlarda): %{boga / (boga + ayi) * 100:.0f}")
    return tablo

stocktwits_duyarlilik("AAPL")

===== AAPL StockTwits duyarlılığı (son 30 mesaj) =====
Etiket
Etiketsiz    21
Bearish       5
Bullish       4
Name: count, dtype: int64 

Boğa oranı (etiketli mesajlarda): %44


,Tarih,Kullanıcı,Takipçi,Etiket,Mesaj
0,2026-11-14 06:59,Diehard777,7,Etiketsiz,$AAPL $SPY believe Apple phones may be collecting personal information from people all over th...
1,2026-11-14 06:29,AcceleratorModel,238,Etiketsiz,$AAPL $NVDA \n\nNVDA — Should snap back to that .2 - .4 positive corr\nSoon\n\nPast 2 weeks ther...
2,2026-11-14 03:20,beestMod516,482,Etiketsiz,Soros portfolio 🤔 \n$AAPL $NVDA $AMZN $GOOG $SPY
3,2026-11-14 02:56,PivotPoint_101Official,5335,Etiketsiz,"$SPY $AAPL $BTC.X\n\nOur traders had a splendid week, \nyour traders are filing for divorce."
4,2026-11-14 02:52,PivotPoint_101Official,5335,Etiketsiz,$SPY $QQQ $AAPL $NVDA \n#pivotgroup💸\n#woofstreets🎬
5,2026-11-14 02:48,SamsonCapital,48250,Etiketsiz,$AMZN $AAPL $NVDA $GOOGL $TKO \n\nSoros Fund Management LLC top holdings:
6,2026-11-14 02:45,LeonardoBonacci,998,Bearish,$QQQ Fib 61.8% rejection played well today. It might not feel like it but this week&#39;s weekl...
7,2026-11-14 02:09,OptionsPlayers,17789,Etiketsiz,$AAPL $QQQ We put in work\nhttps://www.instagram.com/stories/optionsplayers/4029602271980981099?...
8,2026-11-14 02:03,BullseyeGains,12317,Etiketsiz,$AAPL $GOOGL $GOOG $MSFT $AMZN Apple is up 30% this year and now trades at about 38 times forwar...
9,2026-11-14 01:41,earnit_saveit_investit,36,Etiketsiz,$NVDA $TGT $SPOT $AAPL $KR \n \nRECENT CEOs WHO LEFT THEIR COMPANY \nWHERE WILL NVIDIA END UP? ...


In [4]:
import re

def temizle(metin):
    return re.sub(r"http\S+", "", metin).strip()  # linkleri sil

def stocktwits_finbert(sembol):
    tablo = stocktwits_duyarlilik(sembol)
    if tablo is None or tablo.empty:
        return None

    # Tüm mesajları FinBERT'e okut
    esle = {"positive": "Bullish", "negative": "Bearish", "neutral": "Nötr"}
    tahminler = finbert([temizle(m) for m in tablo["Mesaj"]], truncation=True)
    tablo["FinBERT"] = [esle[t["label"]] for t in tahminler]
    tablo["Güven"] = [round(t["score"], 2) for t in tahminler]

    # 1) Model sınavı: Etiketli mesajlarda kullanıcıyla aynı fikirde mi?
    etiketli = tablo[tablo["Etiket"] != "Etiketsiz"]
    if len(etiketli) > 0:
        uyum = (etiketli["Etiket"] == etiketli["FinBERT"]).mean() * 100
        print(f"Model sınavı: FinBERT, etiketli {len(etiketli)} mesajın %{uyum:.0f}'inde kullanıcıyla aynı fikirde\n")

    # 2) Birleşik duyarlılık: Etiket varsa kullanıcınınkini, yoksa FinBERT'inkini al
    tablo["Nihai"] = tablo["Etiket"].where(tablo["Etiket"] != "Etiketsiz", tablo["FinBERT"])
    sayim = tablo["Nihai"].value_counts()
    print("Birleşik duyarlılık:")
    print(sayim)
    boga, ayi = sayim.get("Bullish", 0), sayim.get("Bearish", 0)
    if boga + ayi > 0:
        print(f"\nBoğa oranı: %{boga / (boga + ayi) * 100:.0f} ({boga + ayi} yönlü mesaj üzerinden)")
    return tablo

sonuc = stocktwits_finbert("AAPL")
sonuc[["Etiket", "FinBERT", "Güven", "Mesaj"]]

===== AAPL StockTwits duyarlılığı (son 30 mesaj) =====
Etiket
Etiketsiz    21
Bearish       5
Bullish       4
Name: count, dtype: int64 

Boğa oranı (etiketli mesajlarda): %44
Model sınavı: FinBERT, etiketli 9 mesajın %11'inde kullanıcıyla aynı fikirde

Birleşik duyarlılık:
Nihai
Nötr       18
Bearish     7
Bullish     5
Name: count, dtype: int64

Boğa oranı: %42 (12 yönlü mesaj üzerinden)


,Etiket,FinBERT,Güven,Mesaj
0,Etiketsiz,Nötr,0.55,$AAPL $SPY believe Apple phones may be collecting personal information from people all over th...
1,Etiketsiz,Nötr,0.76,$AAPL $NVDA \n\nNVDA — Should snap back to that .2 - .4 positive corr\nSoon\n\nPast 2 weeks ther...
2,Etiketsiz,Nötr,0.89,Soros portfolio 🤔 \n$AAPL $NVDA $AMZN $GOOG $SPY
3,Etiketsiz,Bearish,0.81,"$SPY $AAPL $BTC.X\n\nOur traders had a splendid week, \nyour traders are filing for divorce."
4,Etiketsiz,Nötr,0.91,$SPY $QQQ $AAPL $NVDA \n#pivotgroup💸\n#woofstreets🎬
5,Etiketsiz,Nötr,0.94,$AMZN $AAPL $NVDA $GOOGL $TKO \n\nSoros Fund Management LLC top holdings:
6,Bearish,Nötr,0.60,$QQQ Fib 61.8% rejection played well today. It might not feel like it but this week&#39;s weekl...
7,Etiketsiz,Nötr,0.89,$AAPL $QQQ We put in work\nhttps://www.instagram.com/stories/optionsplayers/4029602271980981099?...
8,Etiketsiz,Bullish,0.60,$AAPL $GOOGL $GOOG $MSFT $AMZN Apple is up 30% this year and now trades at about 38 times forwar...
9,Etiketsiz,Nötr,0.83,$NVDA $TGT $SPOT $AAPL $KR \n \nRECENT CEOs WHO LEFT THEIR COMPANY \nWHERE WILL NVIDIA END UP? ...


In [5]:
# Sosyal medya için eğitilmiş modeli yükle
fintwit = pipeline("text-classification", model="StephanAkkerman/FinTwitBERT-sentiment")

def etiket_cevir(label):
    label = label.upper()
    if "BULL" in label or "POS" in label: return "Bullish"
    if "BEAR" in label or "NEG" in label: return "Bearish"
    return "Nötr"

mesajlar = [temizle(m) for m in sonuc["Mesaj"]]
sonuc["FinTwit"] = [etiket_cevir(t["label"]) for t in fintwit(mesajlar, truncation=True)]

# Model sınavı: Etiketli mesajlarda iki model kullanıcıyla ne kadar uyumlu?
etiketli = sonuc[sonuc["Etiket"] != "Etiketsiz"]
for model in ["FinBERT", "FinTwit"]:
    uyum = (etiketli["Etiket"] == etiketli[model]).mean() * 100
    print(f"{model:8}: etiketli mesajlarda uyum %{uyum:.0f}")

# Etiketsiz mesajlarda iki model ne diyor?
etiketsiz = sonuc[sonuc["Etiket"] == "Etiketsiz"]
print("\nEtiketsiz mesajlarda dağılım:")
print(pd.DataFrame({
    "FinBERT": etiketsiz["FinBERT"].value_counts(),
    "FinTwit": etiketsiz["FinTwit"].value_counts(),
}).fillna(0).astype(int))

sonuc[["Etiket", "FinBERT", "FinTwit", "Mesaj"]]


config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  439MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/713k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

FinBERT : etiketli mesajlarda uyum %11
FinTwit : etiketli mesajlarda uyum %56

Etiketsiz mesajlarda dağılım:
         FinBERT  FinTwit
Bearish        2        7
Bullish        1        8
Nötr          18        6


,Etiket,FinBERT,FinTwit,Mesaj
0,Etiketsiz,Nötr,Bearish,$AAPL $SPY believe Apple phones may be collecting personal information from people all over th...
1,Etiketsiz,Nötr,Bullish,$AAPL $NVDA \n\nNVDA — Should snap back to that .2 - .4 positive corr\nSoon\n\nPast 2 weeks ther...
2,Etiketsiz,Nötr,Bullish,Soros portfolio 🤔 \n$AAPL $NVDA $AMZN $GOOG $SPY
3,Etiketsiz,Bearish,Bearish,"$SPY $AAPL $BTC.X\n\nOur traders had a splendid week, \nyour traders are filing for divorce."
4,Etiketsiz,Nötr,Bullish,$SPY $QQQ $AAPL $NVDA \n#pivotgroup💸\n#woofstreets🎬
5,Etiketsiz,Nötr,Nötr,$AMZN $AAPL $NVDA $GOOGL $TKO \n\nSoros Fund Management LLC top holdings:
6,Bearish,Nötr,Bearish,$QQQ Fib 61.8% rejection played well today. It might not feel like it but this week&#39;s weekl...
7,Etiketsiz,Nötr,Nötr,$AAPL $QQQ We put in work\nhttps://www.instagram.com/stories/optionsplayers/4029602271980981099?...
8,Etiketsiz,Bullish,Bullish,$AAPL $GOOGL $GOOG $MSFT $AMZN Apple is up 30% this year and now trades at about 38 times forwar...
9,Etiketsiz,Nötr,Bullish,$NVDA $TGT $SPOT $AAPL $KR \n \nRECENT CEOs WHO LEFT THEIR COMPANY \nWHERE WILL NVIDIA END UP? ...


In [6]:
def sosyal_duyarlilik(sembol):
    """StockTwits mesajlarını çeker; kullanıcı etiketi yoksa FinTwit ile tahmin eder."""
    tablo = stocktwits_duyarlilik(sembol)
    if tablo is None or tablo.empty:
        return None

    tahmin = fintwit([temizle(m) for m in tablo["Mesaj"]], truncation=True)
    tablo["Model"] = [etiket_cevir(t["label"]) for t in tahmin]
    tablo["Güven"] = [round(t["score"], 2) for t in tahmin]
    tablo["Nihai"] = tablo["Etiket"].where(tablo["Etiket"] != "Etiketsiz", tablo["Model"])

    sayim = tablo["Nihai"].value_counts()
    boga, ayi = sayim.get("Bullish", 0), sayim.get("Bearish", 0)
    print("\n--- Birleşik sonuç (kullanıcı etiketi + FinTwit) ---")
    print(sayim)
    if boga + ayi > 0:
        print(f"Boğa oranı: %{boga / (boga + ayi) * 100:.0f} ({boga + ayi} yönlü mesaj)")
    return tablo

sosyal_duyarlilik("TSLA")

===== TSLA StockTwits duyarlılığı (son 30 mesaj) =====
Etiket
Etiketsiz    18
Bullish       9
Bearish       3
Name: count, dtype: int64 

Boğa oranı (etiketli mesajlarda): %75

--- Birleşik sonuç (kullanıcı etiketi + FinTwit) ---
Nihai
Bullish    13
Bearish    12
Nötr        5
Name: count, dtype: int64
Boğa oranı: %52 (25 yönlü mesaj)


,Tarih,Kullanıcı,Takipçi,Etiket,Mesaj,Model,Güven,Nihai
0,2026-11-14 07:35,BigBreakingWire,6094,Etiketsiz,Musk Says Tesla Cybercab&#39;s $299 Monthly Subscription Fee Is Too High\n\nTesla has opened Cyb...,Bearish,0.67,Bearish
1,2026-11-14 07:32,soooooooo000OOO000ooooor,6,Etiketsiz,$DJT $NVDA $SPCX $SPY $TSLA,Nötr,0.97,Nötr
2,2026-11-14 06:45,gaineous,760,Etiketsiz,From Ray:\n\n“What if Jensen is the spy?” J/K; stuck at TEB but it’s better than DJT. lol\nGet...,Bullish,0.46,Bullish
3,2026-11-14 06:43,Roth_Trader,1143,Etiketsiz,This past week of epic drama in the markets ($NVDA the lil&#39; rascal) I was able to fade away ...,Bearish,0.70,Bearish
4,2026-11-14 06:35,RiseofTheBull,24,Bullish,$TSLA so are we dipping of ripping on the cybercab news on Monday?,Bearish,0.62,Bullish
5,2026-11-14 06:09,StocktwitsNews,113419,Etiketsiz,Tesla Cybercab Now Available For Purchase For Select Customers — But There’s A Catch\n\n$TSLA $S...,Nötr,0.68,Nötr
6,2026-11-14 05:40,HerrBroekman,75,Etiketsiz,"$SPCX $TSLA social media isnt the real world. In the real world, people are disgusted by your id...",Bearish,0.99,Bearish
7,2026-11-14 05:37,SilverTrader47,212,Bullish,@Ph777trader $GT $NVDA $TSLA \nGoodyear should hire Jensen Huang as its next CEO. An electrica...,Bullish,0.97,Bullish
8,2026-11-14 05:37,SilverTrader47,212,Bullish,@bobvu $GT $NVDA $TSLA \nGoodyear should hire Jensen Huang as its next CEO. An electrical engi...,Bullish,0.97,Bullish
9,2026-11-14 04:48,Mrbullsociety,145,Etiketsiz,TODAY’S RECAP 🔥\n🟢 $TSLA 395C +380%\n🟢 $MSFT 515P +50%\n🟢 $QQQ 685P +500% if you held\n🟢 $SPY 75...,Bullish,0.81,Bullish


In [7]:
import time
from datetime import datetime, timedelta, timezone

def stocktwits_mesajlar(sembol, saat=24, max_sayfa=20):
    url = f"https://api.stocktwits.com/api/2/streams/symbol/{sembol}.json"
    basliklar = {"User-Agent": "Mozilla/5.0"}
    sinir = datetime.now(timezone.utc) - timedelta(hours=saat)
    satirlar, imlec, bitti = [], None, False

    for sayfa in range(max_sayfa):
        params = {"max": imlec} if imlec else {}
        yanit = requests.get(url, headers=basliklar, params=params, timeout=15)
        if yanit.status_code != 200:
            print(f"Sayfa {sayfa + 1}: erişim reddedildi (kod {yanit.status_code}), eldeki verilerle devam.")
            break
        mesajlar = yanit.json().get("messages", [])
        if not mesajlar:
            break

        for m in mesajlar:
            zaman = datetime.fromisoformat(m["created_at"].replace("Z", "+00:00"))
            if zaman < sinir:          # 24 saatten eski mesaja ulaştık
                bitti = True
                break
            etiket = (m.get("entities", {}).get("sentiment") or {}).get("basic")
            satirlar.append({
                "Zaman": zaman,
                "Kullanıcı": m.get("user", {}).get("username", ""),
                "Etiket": etiket or "Etiketsiz",
                "Mesaj": m.get("body", ""),
            })
        if bitti:
            break
        imlec = mesajlar[-1]["id"] - 1   # bir sonraki sayfa: bundan eski mesajlar
        time.sleep(1)                    # sunucuyu yormamak için kısa bekleme

    tablo = pd.DataFrame(satirlar)
    if tablo.empty:
        print("Mesaj bulunamadı.")
        return tablo
    kapsanan = (datetime.now(timezone.utc) - tablo["Zaman"].min()).total_seconds() / 3600
    print(f"{sembol}: {len(tablo)} mesaj, son {kapsanan:.1f} saati kapsıyor")
    if not bitti:
        print(f"Uyarı: Sayfa sınırına ulaşıldı, {saat} saatin tamamı kapsanamadı.")
    return tablo


def sosyal_duyarlilik(sembol, saat=24, max_sayfa=20):
    tablo = stocktwits_mesajlar(sembol, saat, max_sayfa)
    if tablo.empty:
        return None

    tahmin = fintwit([temizle(m) for m in tablo["Mesaj"]], truncation=True, batch_size=32)
    tablo["Model"] = [etiket_cevir(t["label"]) for t in tahmin]
    tablo["Nihai"] = tablo["Etiket"].where(tablo["Etiket"] != "Etiketsiz", tablo["Model"])

    sayim = tablo["Nihai"].value_counts()
    boga, ayi = sayim.get("Bullish", 0), sayim.get("Bearish", 0)
    print(sayim)
    if boga + ayi > 0:
        print(f"Boğa oranı: %{boga / (boga + ayi) * 100:.0f} ({boga + ayi} yönlü mesaj)")
    return tablo

sonuc = sosyal_duyarlilik("AAPL", saat=12)

AAPL: 402 mesaj, son 23.5 saati kapsıyor
Nihai
Bearish    211
Bullish    154
Nötr        37
Name: count, dtype: int64
Boğa oranı: %42 (365 yönlü mesaj)
